# KisanAI - Crop Recommendation ML

KisanAI / Sampoorn Kisan AI Sahayak

## Goal
Train and compare multiple ML models for crop recommendation using:
N, P, K, temperature, humidity, pH, rainfall → crop label.

This notebook is intentionally organized as an experiment: inspect data → preprocess → train → evaluate → save the best model.

In [ ]:
# Colab setup
!pip -q install pandas numpy matplotlib seaborn scikit-learn joblib

## Upload the extracted crop dataset
If you have `Crop rec.zip`, upload it below, or upload the CSV directly.
For easiest use, upload the CSV extracted from the ZIP.

In [ ]:
from google.colab import files
uploaded = files.upload()

import os
print("Uploaded:", list(uploaded.keys()))

In [ ]:
import pandas as pd, os, glob

csvs = glob.glob("/content/**/*.csv", recursive=True)
print("CSV files found:")
for f in csvs[:20]:
    print(f)

# Change this after seeing the printed list if needed
CSV_PATH = csvs[0]
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

target_candidates = [c for c in df.columns if c.lower() in ["label", "crop", "crop_name", "target"]]
print("Possible target columns:", target_candidates)

TARGET = target_candidates[0] if target_candidates else df.columns[-1]
print("Using target:", TARGET)

print(df[TARGET].value_counts().head(30))

## Prepare X and y
The code below automatically uses numeric columns as model features. If the dataset contains an ID or other numeric column that should not be used, remove it here.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=[TARGET])
y = df[TARGET]

# Keep numeric columns for the first experiment
X = X.select_dtypes(include="number")

print("Features:", X.columns.tolist())
print("X shape:", X.shape)
print("y classes:", y.nunique())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

## Train and compare models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True))
    ])
}

results = []
trained = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    p, r, f1, _ = precision_recall_fscore_support(
        y_test, pred, average="weighted", zero_division=0
    )
    acc = accuracy_score(y_test, pred)
    results.append([name, acc, p, r, f1])
    trained[name] = model

results_df = pd.DataFrame(
    results, columns=["Model","Accuracy","Precision","Recall","F1"]
).sort_values("Accuracy", ascending=False)

results_df

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained[best_name]

print("Best model:", best_name)
print(classification_report(y_test, best_model.predict(X_test), zero_division=0))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test, best_model.predict(X_test), xticks_rotation=90
)
plt.tight_layout()
plt.show()

## Save the trained model
The saved `.pkl` can later be loaded by the Crop Recommendation Agent / FastAPI backend.

In [ ]:
import joblib, os

os.makedirs("/content/kisanai_models", exist_ok=True)

artifact = {
    "model": best_model,
    "features": X.columns.tolist(),
    "target": TARGET
}

MODEL_PATH = "/content/kisanai_models/crop_recommendation_model.pkl"
joblib.dump(artifact, MODEL_PATH)

print("Saved:", MODEL_PATH)

In [ ]:
# Test with one real row from the test set
sample = X_test.iloc[[0]]
prediction = best_model.predict(sample)[0]

print("Input:")
display(sample)
print("Predicted crop:", prediction)

if hasattr(best_model, "predict_proba"):
    probs = best_model.predict_proba(sample)[0]
    classes = best_model.classes_
    top = sorted(zip(classes, probs), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop predictions:")
    for crop, prob in top:
        print(f"{crop}: {prob:.2%}")

## Download model for the local KisanAI project

In [ ]:
from google.colab import files
files.download(MODEL_PATH)